In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import matplotlib.patches as patches


# ==============================================================================
# CELL 1: CONFIGURATION & STYLE
# ==============================================================================
def matplotlib_default_config():
    """Returns the default dictionary configurations for publication-ready plots."""
    font = {'family': 'serif', 'weight': 'normal', 'size': 14}
    rcnew = {
        "mathtext.fontset": "cm",
        "text.usetex": False,
        'figure.frameon': True,
        'axes.linewidth': 2.,
        "axes.titlesize": 20,
        "axes.labelsize": 20,
        "legend.fontsize": 12,
        'legend.fancybox': True,
        'lines.linewidth': 2.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.top': True,
        'ytick.right': True,
        'xtick.labelsize': 18,
        'ytick.labelsize': 18,
    }
    return font, rcnew

%matplotlib widget
font, rcnew = matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)


In [ ]:
# Intermidiate cell to tell where the new data are: 

from pathlib import Path

# Sobe a partir do cwd até achar a raiz do projeto (a pasta que contém tools/)
PROJ_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools").is_dir())

DATA_DIR = PROJ_ROOT / "data_TNG50-2_snap025_5runs"   # name that the launcher created
#GRID_DIR = Path.cwd() / "data_grid"                    # continua em notebooks_TNG/

print("project :", PROJ_ROOT)
print("data   :", DATA_DIR, "|", len(list(DATA_DIR.glob('*.pkl'))), "pkl found")

In [ ]:
# ==============================================================================
# CELL 2: LOAD GRID DATA & COMPUTE GLOBAL FRACTIONS
# ==============================================================================
grid_file_path = 'data_grid/TNG50-2_z3_Grid_500x500_DLAMode.pkl'

print("Loading the Grid file...")
df_grid = pd.read_pickle(grid_file_path)

# Binning thresholds for absorber categories
bins_grid   = [0, 1.6e17, 1e19, 2e20, 1e21, np.inf]
labels_grid = ['Forest', 'LLS', 'Sub_DLA', 'Small_DLA', 'Large_DLA']

df_grid['category'] = pd.cut(
    df_grid['total_col_den'], bins=bins_grid, labels=labels_grid, right=False
)

total_grid_skewers = len(df_grid)
grid_counts    = df_grid['category'].value_counts(sort=False)
grid_fractions = grid_counts / total_grid_skewers

print(f"Grid Data Processed: {total_grid_skewers:,} skewers.")

In [ ]:
# ==============================================================================
# CELL 3: LOAD & PREPARE ALL VARIANCE RUNS (sem Extended)
# ==============================================================================
data_dir = DATA_DIR

# --- Discover run files (exclude anything with "Extended_") ---
orig_pattern = os.path.join(data_dir, 'TNG50-2_z3_*_HCDMode_Stratified_run*.pkl')
run_files    = sorted(glob.glob(orig_pattern))
run_files    = [f for f in run_files if 'Extended_' not in os.path.basename(f)]

if len(run_files) == 0:
    raise FileNotFoundError(f"No run files found in {data_dir}!")



# If everything works, we go from here now...
print(f"Found {len(run_files)} run files (Extended ignored).")

# --- Load each run ---
dfs_runs = []

for rf in run_files:
    df = pd.read_pickle(rf)
    df['r_mult_round'] = df['r_multiplier'].round(2)
    dfs_runs.append(df)

    basename = os.path.basename(rf)
    print(f"  {basename}: {len(df):>7,} rows")

# All runs should share the same set of impact parameters
unique_radii = np.sort(dfs_runs[0]['r_mult_round'].unique())
n_runs       = len(dfs_runs)

print(f"\n{n_runs} runs loaded.  Impact parameters ({len(unique_radii)}):")
print(f"  {unique_radii}")

In [ ]:
len(dfs_runs[0])

In [ ]:
# ==============================================================================
# CELL 4: CALCULATE FRACTIONS & VARIANCE PER IMPACT PARAMETER
# ==============================================================================
cat_styles = {
    'Large_DLA': {'color': 'indigo',       'marker': 'v',
                  'label': r'Large DLA ($\log N_{\mathrm{HI}} \geq 21.0$)'},
    'Small_DLA': {'color': 'darkred',      'marker': 'o',
                  'label': r'Small DLA ($20.3 \leq \log N_{\mathrm{HI}} < 21.0$)'},
    'Sub_DLA':   {'color': 'darkorange',   'marker': 's',
                  'label': r'Sub-DLA ($19.0 \leq \log N_{\mathrm{HI}} < 20.3$)'},
    'LLS':       {'color': 'forestgreen',  'marker': '^',
                  'label': r'LLS ($17.2 \leq \log N_{\mathrm{HI}} < 19.0$)'},
    'Forest':    {'color': 'midnightblue', 'marker': 'D',
                  'label': r'Ly-$\alpha$ Forest ($\log N_{\mathrm{HI}} < 17.2$)'},
}

# Collect fraction profiles: shape will be (n_runs, n_radii) per category
fractions_raw = {cat: [] for cat in cat_styles}

for df_run in dfs_runs:
    run_fracs = {cat: [] for cat in cat_styles}

    for b_val in unique_radii:
        df_b = df_run[df_run['r_mult_round'] == b_val]
        tot  = len(df_b)

        if tot > 0:
            nhi = df_b['log_n_hi']
            run_fracs['Large_DLA'].append(len(nhi[nhi >= 21.0]) / tot)
            run_fracs['Small_DLA'].append(len(nhi[(nhi >= 20.3) & (nhi < 21.0)]) / tot)
            run_fracs['Sub_DLA'].append(len(nhi[(nhi >= 19.0) & (nhi < 20.3)]) / tot)
            run_fracs['LLS'].append(len(nhi[(nhi >= 17.2) & (nhi < 19.0)]) / tot)
            run_fracs['Forest'].append(len(nhi[nhi < 17.2]) / tot)
        else:
            for cat in cat_styles:
                run_fracs[cat].append(0.0)

    for cat in cat_styles:
        fractions_raw[cat].append(run_fracs[cat])

# Convert to numpy arrays for vectorized stats
for cat in cat_styles:
    fractions_raw[cat] = np.array(fractions_raw[cat])

In [ ]:
# ==============================================================================
# CELL 5: COMBINED PLOT (PROFILES + GRID PANEL, ±3σ SHADING, UP TO 20 R200)
# ==============================================================================
NSIGMA = 3  # <-- Change this single number for 1σ, 2σ, etc.

fig, (ax_main, ax_grid) = plt.subplots(1, 2, figsize=(13, 7.5), sharey=True, gridspec_kw={'width_ratios': [4, 1], 'wspace': 0.05})

for cat, style in cat_styles.items():
    data_matrix = fractions_raw[cat]  # (n_runs, n_radii)

    mean_frac = np.mean(data_matrix, axis=0)
    std_frac  = np.std(data_matrix, axis=0)

    # ±Nσ shading on main panel
    ax_main.fill_between(
        unique_radii,
        np.clip(mean_frac - NSIGMA * std_frac, 0, None),
        np.clip(mean_frac + NSIGMA * std_frac, None, 1),
        color=style['color'], alpha=0.20, zorder=1,
    )

    # Individual runs (thin, faint)
    for run_idx in range(data_matrix.shape[0]):
        ax_main.plot(
            unique_radii, data_matrix[run_idx, :],
            color=style['color'], alpha=0.20, lw=0.8, ls='-', zorder=2,
        )

    # Mean line (bold markers)
    ax_main.plot(
        unique_radii, mean_frac,
        color=style['color'], lw=2.5, ls='-',
        marker=style['marker'], ms=5, mec='white', zorder=3,
    )

# --- Side panel: Grid averages & variance extension ---
x_pos_grid = 1.0

for cat, style in cat_styles.items():
    grid_val = grid_fractions[cat]

    data_matrix = fractions_raw[cat]
    last_mean = np.mean(data_matrix, axis=0)[-1]
    last_std  = np.std(data_matrix, axis=0)[-1]

    # Extend the ±Nσ band into the side panel
    ax_grid.axhspan(
        np.clip(last_mean - NSIGMA * last_std, 0, None),
        np.clip(last_mean + NSIGMA * last_std, None, 1),
        color=style['color'], alpha=0.20, zorder=1,
    )

    # Grid marker
    ax_grid.plot(
        x_pos_grid, grid_val,
        color=style['color'], marker=style['marker'],
        ms=5, mec='white', zorder=3,
    )

    # Cosmic-mean dotted lines spanning both panels
    ax_main.axhline(grid_val, color=style['color'], ls=':', alpha=0.3, zorder=0)
    ax_grid.axhline(grid_val, color=style['color'], ls=':', alpha=0.3, zorder=0)

# --- Aesthetics: main panel ---
ax_main.grid(True, ls='--', alpha=0.5)
ax_main.set_xscale('log')
ax_main.set_ylim(-0.02, 1.05)
ax_main.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

ax_main.set_xticks([0.01, 0.1, 1.0, 5.0, 10.0, 20.0, 50.0])
ax_main.set_xticklabels(['0.01', '0.1', '1', '5', '10', '20', '50'])
ax_main.set_xlim(unique_radii[unique_radii > 0].min() * 0.7, 50)

ax_main.set_xlabel(r'Impact Parameter $b \ [R_{200}]$')
ax_main.set_ylabel(r'Covering Fraction $f_C(b)$')
ax_main.set_title(r'Absorber Covering Fraction (TNG50-2)', pad=15)

# --- Aesthetics: side panel ---
ax_grid.grid(True, ls='--', alpha=0.5)
ax_grid.set_xlim(0.8, 1.2)
ax_grid.set_xticks([x_pos_grid])
ax_grid.set_xticklabels(['Grid\nAverage'])
ax_grid.set_title('Global', pad=15)

# --- Composite legend ---
legend_elements = [
    Line2D([0], [0], color=c['color'], lw=2.5, marker=c['marker'], label=c['label'])
    for c in cat_styles.values()
]
legend_elements.append(Line2D([0], [0], color='none', label=' '))
legend_elements.append(
    patches.Patch(
        facecolor='gray', alpha=0.3, edgecolor='gray',
        label=rf'$\pm {NSIGMA}\sigma$ Variance ({n_runs} runs)',
    )
)
legend_elements.append(
    Line2D([0], [0], color='gray', ls=':', alpha=0.5, lw=2,
           label='Cosmic Mean (Grid)')
)

ax_main.legend(
    handles=legend_elements, loc='center right', frameon=True,
    fontsize=10.5, edgecolor='lightgray', facecolor='white', framealpha=0.9,
)

plt.tight_layout()
plt.show()

In [ ]:
# Plot without the side panel for better visibility of the main panel! 

# ==============================================================================
# CELL 5b: COVERING FRACTION PROFILE ONLY (±3σ SHADING)
# ==============================================================================
NSIGMA = 3  # <-- Change this single number for 1σ, 2σ, etc.

fig, ax_main = plt.subplots(figsize=(10, 8))

for cat, style in cat_styles.items():
    data_matrix = fractions_raw[cat]  # (n_runs, n_radii)

    mean_frac = np.mean(data_matrix, axis=0)
    std_frac  = np.std(data_matrix, axis=0)

    # ±Nσ shading
    ax_main.fill_between(
        unique_radii,
        np.clip(mean_frac - NSIGMA * std_frac, 0, None),
        np.clip(mean_frac + NSIGMA * std_frac, None, 1),
        color=style['color'],
        alpha=0.20,
        zorder=1,
    )

    # Individual runs: thin and faint
    for run_idx in range(data_matrix.shape[0]):
        ax_main.plot(
            unique_radii,
            data_matrix[run_idx, :],
            color=style['color'],
            alpha=0.20,
            lw=0.8,
            ls='-',
            zorder=2,
        )

    # Mean line
    ax_main.plot(
        unique_radii,
        mean_frac,
        color=style['color'],
        lw=2.5,
        ls='-',
        marker=style['marker'],
        ms=5,
        mec='white',
        zorder=3,
    )

# --- Aesthetics ---
ax_main.grid(True, ls='--', alpha=0.5)
ax_main.set_xscale('log')
ax_main.set_ylim(-0.02, 1.05)
ax_main.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))

ax_main.set_xticks([0.01, 0.1, 1.0, 5.0, 10.0, 20.0, 50.0])
ax_main.set_xticklabels(['0.01', '0.1', '1', '5', '10', '20', '50'])
ax_main.set_xlim(unique_radii[unique_radii > 0].min() * 0.7, 50)

ax_main.set_xlabel(r'Impact Parameter $b \ [R_{200}]$')
ax_main.set_ylabel(r'Covering Fraction $f_C(b)$')
ax_main.set_title(r'Absorber Covering Fraction (TNG50-2)', pad=15)

# --- Legend ---
legend_elements = [
    Line2D(
        [0], [0],
        color=c['color'],
        lw=2.5,
        marker=c['marker'],
        label=c['label'],
    )
    for c in cat_styles.values()
]

legend_elements.append(Line2D([0], [0], color='none', label=' '))

legend_elements.append(
    patches.Patch(
        facecolor='gray',
        alpha=0.3,
        edgecolor='gray',
        label=rf'$\pm {NSIGMA}\sigma$ variance ({n_runs} runs)',
    )
)

ax_main.legend(
    handles=legend_elements,
    loc='center right',
    frameon=True,
    fontsize=10.5,
    edgecolor='lightgray',
    facecolor='white',
    framealpha=0.9,
)

plt.tight_layout()
plt.show()

# Ridgeline Plots!

In [ ]:
# Ridgeline Plots!
# ==============================================================================
# CELL 7: RIDGELINE — VARIANCE ACROSS RUNS (DENSITY-NORMALIZED, ALIGNED EDGES)
# ==============================================================================

# --------------------------------------------------------------------------
# 0. Configuration
# --------------------------------------------------------------------------
HUBBLE_H = 0.6774
MASS_CONVERSION_FACTOR = 1e10 / HUBBLE_H

for df_run in dfs_runs:
    if 'halo_mass_msun' not in df_run.columns:
        df_run['halo_mass_msun'] = df_run['halo_mass'] * MASS_CONVERSION_FACTOR

# Filter impact parameters: keep 0.0, up to 2.0 R200
SELECTED_B = sorted(unique_radii[unique_radii >= 0.0])
SELECTED_B = [b for b in SELECTED_B if b <= 2.0]
print(f"Impact parameters for ridgeline ({len(SELECTED_B)}):\n  {SELECTED_B}")

# Mass bins
mass_edges = [10**10.0, 10**10.5, 10**11.0, 10**11.5, 10**12.0, 10**13.0]
mass_labels = [
    r'$10^{10.0} \leq M_{200} < 10^{10.5} \; M_\odot$',
    r'$10^{10.5} \leq M_{200} < 10^{11.0} \; M_\odot$',
    r'$10^{11.0} \leq M_{200} < 10^{11.5} \; M_\odot$',
    r'$10^{11.5} \leq M_{200} < 10^{12.0} \; M_\odot$',
    r'$10^{12.0} \leq M_{200} < 10^{13.0} \; M_\odot$',
]
n_mass_bins = len(mass_labels)

# Background region colours
c_forest   = '#E3F2FD'
c_lls      = '#E8F5E9'
c_subdla   = '#FFF3E0'
c_smalldla = '#FFCDD2'
c_largedla = '#E1BEE7'

# Run colours
run_colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange', 'tab:purple']

# ----------------------------------------------------------------------------
# overlap_factor: COSMETIC VERTICAL SCALING.
# Each ridgeline curve is drawn above an integer y-base. With density-per-dex
# normalization the natural curve heights are small (~0.5-2.0), so we multiply
# by overlap_factor to make the curves visually prominent. It does NOT distort
# relative areas: the area between two log N_HI limits equals the fraction of
# skewers in that interval (exactly the covering-fraction quantity).
# Typical good values: 2.0 - 4.0. Bigger = more overlap between rows.
# ----------------------------------------------------------------------------
overlap_factor = 1.8
NSIGMA_RIDGE   = 3    # 1 standard deviation
n_sel          = len(SELECTED_B)

# Histogram definitions (restricted to 16.0 -> 23.0)
num_bins  = 100
hist_bins = np.linspace(16.0, 23.0, num_bins + 1)
x_hist    = (hist_bins[:-1] + hist_bins[1:]) / 2
bin_width = hist_bins[1] - hist_bins[0]   # constant since we use linspace


# --------------------------------------------------------------------------
# CORRECTED HISTOGRAM: density per dex (dn / d log N_HI / N_total)
# --------------------------------------------------------------------------
def density_hist(values, bins, tot_skewers):
    """
    Returns probability density per unit log N_HI.
    
    Height in each bin = (counts_in_bin / total_skewers) / bin_width
    
    Properties:
    - Integral over the entire range = fraction of skewers with valid log_n_hi.
    - Integral over any sub-range [a, b] = fraction of skewers with log N_HI in [a, b].
    - Comparable between distributions and across simulations.
    """
    if len(values) < 2 or tot_skewers == 0:
        return np.zeros(len(bins) - 1)
    
    counts, _ = np.histogram(values, bins=bins)
    bw = bins[1] - bins[0]
    return counts / (tot_skewers * bw)


# --------------------------------------------------------------------------
# Generic ridgeline painter (density-normalized + aligned category edges)
# --------------------------------------------------------------------------
def paint_ridgeline(ax, dfs_runs, selected_b, mass_low=None, mass_high=None,
                    title_text='', show_legend=True):

    # Background spans — boundary at 21.0 (CONSISTENT with covering fraction)
    ax.axvspan(16.0, 17.2, color=c_forest,   zorder=0)
    ax.axvspan(17.2, 19.0, color=c_lls,      zorder=0)
    ax.axvspan(19.0, 20.3, color=c_subdla,   zorder=0)
    ax.axvspan(20.3, 21.0, color=c_smalldla, zorder=0)
    ax.axvspan(21.0, 23.0, color=c_largedla, zorder=0)

    # Threshold markers (now at 21.0 instead of 21.5)
    for xv in [17.2, 19.0, 20.3, 21.0]:
        ax.axvline(xv, color='white', ls='--', lw=2.0, zorder=0)

    # Regime Text Boxes at the top of the plot
    text_y_pos = 0.98
    bbox_props = dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7, edgecolor="none")
    ax.text(16.60, text_y_pos, r'Ly-$\alpha$ Forest', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=10, bbox=bbox_props, zorder=10)
    ax.text(18.10, text_y_pos, 'LLS', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=10, bbox=bbox_props, zorder=10)
    ax.text(19.65, text_y_pos, 'Sub-DLA', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=10, bbox=bbox_props, zorder=10)
    ax.text(20.65, text_y_pos, 'Small-DLA', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=10, bbox=bbox_props, zorder=10)
    ax.text(22.00, text_y_pos, 'Large-DLA', transform=ax.get_xaxis_transform(),
            ha='center', va='top', fontsize=10, bbox=bbox_props, zorder=10)

    ax.grid(axis='x', ls=':', alpha=0.4)
    ax.grid(axis='y', visible=False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    y_ticks  = []
    y_labels = []

    for i_b, b_val in enumerate(selected_b):
        b_round = np.round(b_val, 2)
        y_base  = i_b

        y_ticks.append(y_base)
        y_labels.append(f"{b_val:.2f}")
        ax.axhline(y_base, color='gray', lw=0.8, alpha=0.3, zorder=1)

        hist_stack  = []
        forest_pcts = []

        for r_idx, df_run in enumerate(dfs_runs):
            if mass_low is not None and mass_high is not None:
                df_mass = df_run[
                    (df_run['halo_mass_msun'] >= mass_low) &
                    (df_run['halo_mass_msun'] <  mass_high)
                ]
            else:
                df_mass = df_run

            df_b   = df_mass[df_mass['r_mult_round'] == b_round]
            tot    = len(df_b)
            
            # Pipeline forces sub-threshold to 0.0, so valid contains only HCDs.
            valid  = df_b[df_b['log_n_hi'] > 1.0]['log_n_hi'].values

            # Density per dex, normalized by TOTAL skewers (including forest)
            mask_hist = (valid >= 16.0) & (valid <= 23.0)
            hist_dens = density_hist(valid[mask_hist], hist_bins, tot)
            hist_stack.append(hist_dens)

            # Forest fraction
            if tot > 0:
                pct_all = len(df_b[df_b['log_n_hi'] < 17.2]) / tot * 100
            else:
                pct_all = 0.0
                
            forest_pcts.append(pct_all)

        hist_stack = np.array(hist_stack)

        # --- Plot individual runs as thin transparent lines ---
        for r_idx in range(len(dfs_runs)):
            scaled = hist_stack[r_idx] * overlap_factor
            
            ax.plot(
                x_hist, y_base + scaled,
                color=run_colors[r_idx % len(run_colors)],
                lw=0.8, alpha=0.4, ls='-', zorder=i_b + 3,
            )

        # --- Mean ± Nσ envelope ---
        mean_hist = np.mean(hist_stack, axis=0)
        std_hist  = np.std(hist_stack, axis=0)

        mean_scaled = mean_hist * overlap_factor
        lo_scaled   = np.clip(mean_hist - NSIGMA_RIDGE * std_hist, 0, None) * overlap_factor
        hi_scaled   = (mean_hist + NSIGMA_RIDGE * std_hist) * overlap_factor

        # Shading
        ax.fill_between(
            x_hist, y_base + lo_scaled, y_base + hi_scaled,
            color='gray', alpha=0.20, zorder=i_b + 4,
        )

        # Mean line (solid black)
        ax.plot(
            x_hist, y_base + mean_scaled,
            color='black', lw=1.6, ls='-', zorder=i_b + 6,
        )

        # Forest fraction annotation
        mean_f   = np.mean(forest_pcts)
        std_f    = np.std(forest_pcts)
        
        ax.text(
            16.1, y_base + 0.15,
            rf"{mean_f:.0f}$\pm${std_f:.0f}%",
            color='black', fontsize=15, fontweight='bold', zorder=20,
        )

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlim(16.0, 23.0)
    ax.set_ylim(-0.5, len(selected_b) - 1 + overlap_factor + 0.5)

    if title_text:
        ax.text(
            0.98, 0.90, title_text, transform=ax.transAxes,
            fontsize=13, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.9, edgecolor='none'),
            zorder=30,
        )

    if show_legend:
        run_lines = [
            Line2D([0], [0], color=run_colors[r % len(run_colors)],
                   lw=1.0, alpha=0.55, label=f'Run {r}')
            for r in range(len(dfs_runs))
        ]
        mean_line  = Line2D([0], [0], color='black', lw=1.6, label='Mean')
        band_patch = patches.Patch(
            facecolor='gray', alpha=0.25, edgecolor='gray',
            label=rf'$\pm {NSIGMA_RIDGE}\sigma$ ({len(dfs_runs)} runs)',
        )
        spacer = Line2D([0], [0], marker='none', linestyle='none', label=' ')

        all_handles = run_lines + [mean_line, band_patch, spacer]

        leg = ax.legend(
            handles=all_handles, loc='upper right', ncol=2,
            frameon=True, fontsize=15,
            bbox_to_anchor=(0.99, 0.85),
            facecolor='white', framealpha=0.95,
            edgecolor='darkgray', columnspacing=1.0, handletextpad=0.5,
        )
        leg.set_zorder(100)

In [ ]:
# ==========================================================================
# PLOT A: RIDGELINE — ALL MASSES COMBINED
# ==========================================================================
fig_height_all = max(8, 0.60 * n_sel)
fig_all, ax_all = plt.subplots(figsize=(13, fig_height_all))

paint_ridgeline(
    ax_all, dfs_runs, SELECTED_B,
    mass_low=None, mass_high=None,
    title_text='All Masses Combined',
    show_legend=True,
)

ax_all.set_xlabel(r"$\log N_{\rm HI}$ [cm$^{-2}$]")
ax_all.set_ylabel(r'Impact Parameter $b \ [R_{200}]$')
ax_all.set_title(
    rf'TNG50-2 Variance Ridgeline ($\pm{NSIGMA_RIDGE}\sigma$, {n_runs} runs)',
    pad=15,
)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CELL 8: COVERING FRACTION PROFILES SPLIT BY HALO MASS BIN
# ==============================================================================
# Reuses: dfs_runs, unique_radii, cat_styles, n_runs, mass_edges, HUBBLE_H
NSIGMA = 3

# Ensure the physical-mass column exists (idempotent w.r.t. Cell 7)
MASS_CONVERSION_FACTOR = 1e10 / HUBBLE_H
for df_run in dfs_runs:
    if 'halo_mass_msun' not in df_run.columns:
        df_run['halo_mass_msun'] = df_run['halo_mass'] * MASS_CONVERSION_FACTOR

mass_bin_titles = [
    r'$10^{10.0}$–$10^{10.5}\,M_\odot$',
    r'$10^{10.5}$–$10^{11.0}\,M_\odot$',
    r'$10^{11.0}$–$10^{11.5}\,M_\odot$',
    r'$10^{11.5}$–$10^{12.0}\,M_\odot$',
    r'$10^{12.0}$–$10^{13.0}\,M_\odot$',
]
n_mass_bins = len(mass_bin_titles)

# --------------------------------------------------------------------------
# Fraction profiles (n_runs, n_radii) per category for a mass-selected subset.
# Empty (bin, b) cells -> NaN so they are ignored in mean/std and simply not drawn.
# --------------------------------------------------------------------------
def compute_fraction_profiles(dfs, radii, mass_low=None, mass_high=None):
    out = {cat: [] for cat in cat_styles}
    counts = []
    for df_run in dfs:
        if mass_low is not None:
            df_m = df_run[(df_run['halo_mass_msun'] >= mass_low) &
                          (df_run['halo_mass_msun'] <  mass_high)]
        else:
            df_m = df_run

        run_fracs = {cat: [] for cat in cat_styles}
        run_counts = []
        for b_val in radii:
            df_b = df_m[df_m['r_mult_round'] == b_val]
            tot = len(df_b)
            run_counts.append(tot)
            if tot > 0:
                nhi = df_b['log_n_hi']
                run_fracs['Large_DLA'].append(len(nhi[nhi >= 21.0]) / tot)
                run_fracs['Small_DLA'].append(len(nhi[(nhi >= 20.3) & (nhi < 21.0)]) / tot)
                run_fracs['Sub_DLA'].append(len(nhi[(nhi >= 19.0) & (nhi < 20.3)]) / tot)
                run_fracs['LLS'].append(len(nhi[(nhi >= 17.2) & (nhi < 19.0)]) / tot)
                run_fracs['Forest'].append(len(nhi[nhi < 17.2]) / tot)
            else:
                for cat in cat_styles:
                    run_fracs[cat].append(np.nan)

        for cat in cat_styles:
            out[cat].append(run_fracs[cat])
        counts.append(run_counts)

    for cat in cat_styles:
        out[cat] = np.array(out[cat])
    return out, np.array(counts)

# --------------------------------------------------------------------------
# 2x3 grid: 5 mass bins + 1 "All Masses" reference panel
# --------------------------------------------------------------------------
panel_specs = [(mass_edges[i], mass_edges[i+1], mass_bin_titles[i])
               for i in range(n_mass_bins)]
panel_specs.append((None, None, 'All Masses'))

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (m_lo, m_hi, title) in zip(axes, panel_specs):
    profiles, counts = compute_fraction_profiles(dfs_runs, unique_radii, m_lo, m_hi)

    for cat, style in cat_styles.items():
        data = profiles[cat]                      # (n_runs, n_radii)
        with np.errstate(invalid='ignore'):
            mean_frac = np.nanmean(data, axis=0)
            std_frac  = np.nanstd(data, axis=0)

        ax.fill_between(
            unique_radii,
            np.clip(mean_frac - NSIGMA * std_frac, 0, None),
            np.clip(mean_frac + NSIGMA * std_frac, None, 1),
            color=style['color'], alpha=0.18, zorder=1,
        )
        ax.plot(
            unique_radii, mean_frac,
            color=style['color'], lw=2.0, ls='-',
            marker=style['marker'], ms=4, mec='white', zorder=3,
        )

    # Typical sample size per impact parameter (median over populated b, all runs)
    populated = counts[counts > 0]
    med_n = int(np.median(populated)) if populated.size else 0

    ax.text(0.035, 0.97, title, transform=ax.transAxes, ha='left', va='top',
            fontsize=13, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      alpha=0.85, edgecolor='lightgray'), zorder=10)
    ax.text(0.035, 0.85, rf'$\sim${med_n:,} skewers$/b$', transform=ax.transAxes,
            ha='left', va='top', fontsize=9.5, color='dimgray', zorder=10)

    ax.grid(True, ls='--', alpha=0.5)
    ax.set_xscale('log')
    ax.set_ylim(-0.02, 1.05)
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1.0))
    ax.set_xticks([0.01, 0.1, 1.0, 10.0, 50.0])
    ax.set_xticklabels(['0.01', '0.1', '1', '10', '50'])
    ax.set_xlim(unique_radii[unique_radii > 0].min() * 0.7, 50)

# --- Shared labels ---
fig.supylabel(r'Covering Fraction $f_C(b)$', fontsize=20, x=0.02)   # <-- FIX: era x=0.055
fig.suptitle(r'Absorber Covering Fraction by Halo Mass (TNG50-2)',
             fontsize=22, y=0.97)
fig.supxlabel(r'Impact Parameter $b \ [R_{200}]$', fontsize=20, y=0.13)  # <-- FIX: era y=0.115

# --- Single shared legend below everything ---
legend_elements = [
    Line2D([0], [0], color=c['color'], lw=2.5, marker=c['marker'], label=c['label'])
    for c in cat_styles.values()
]
legend_elements.append(
    patches.Patch(facecolor='gray', alpha=0.3, edgecolor='gray',
                  label=rf'$\pm {NSIGMA}\sigma$ variance ({n_runs} runs)')
)
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           frameon=True, fontsize=11, edgecolor='lightgray',
           facecolor='white', framealpha=0.9,
           bbox_to_anchor=(0.5, 0.01))

# Manual margins instead of tight_layout (which fights the supxlabel/legend)
fig.subplots_adjust(left=0.11, right=0.98, top=0.90, bottom=0.20,   # <-- FIX: left era 0.085
                    wspace=0.08, hspace=0.13)

plt.show()

### Let's see now the individual behavior of the absorptions for a choosen halo to a choosen impact parameter

In [ ]:
### Halo overview: HI map + the very same skewers' absorption, in one figure
# ==============================================================================
# CELL 10: COMBINED FIGURE
#   top    -> HI column-density map (z projection) with HCD/DLA contours,
#             R_200, and the 24 sightlines drawn where they actually pass
#   bottom -> the spectra of those same 24 sightlines
# Everything is driven by the knobs in STEP 0, so halo and radii stay free.
# ==============================================================================
import h5py
from scipy.ndimage import gaussian_filter
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ------------------------------------------------------------------
# STEP 0. Every knob in one place
# ------------------------------------------------------------------
# -- halo selection --
SELECT_SEED   = 0          # only used when HALO_ID is None
HALO_ID       = 265        # set to None to draw a random halo from the mass bin
MASS_LO_LOG   = 11.5       # mass bin, log10(M_200 / M_sun), physical
MASS_HI_LOG   = 12.0
# -- impact parameters: (requested b in R_200, colour). Add a third if you like --
B_SETS        = [(0.01, 'tab:blue'),
                 (0.2,  'tab:red')]
# -- bottom panel --
SHOW_MEDIAN   = False      # thick line = median of each set
VEL_WINDOW    = 2000       # x half-width in km/s (None = full box)
# -- map / snapshot --
SIM_NAME      = 'TNG50-2'
SNAP_NUM      = 25
STORAGE       = f'/home/STORAGE/{SIM_NAME}'
BOX_HALF_R200 = 3.0        # map half-width, in units of R_200
NPIX          = 256        # pixels per side
SMOOTH_SIGMA  = 1.0        # Gaussian smoothing in pixels (0 = off)
X_HYDROGEN    = 0.76       # hydrogen mass fraction, same value the pipeline uses
MAX_CHUNKS    = None       # e.g. 8 for a quick test; None = every chunk
USE_CACHE     = True       # skip re-reading the snapshot when only tweaking the plot
# -- constants --
MSUN_G, MPROTON_G, KPC_CM = 1.989e33, 1.6726e-24, 3.0857e21

# ------------------------------------------------------------------
# STEP 1. Take one run and pick the halo inside the mass bin
# ------------------------------------------------------------------
df_run = dfs_runs[0]
if 'halo_mass_msun' not in df_run.columns:
    df_run['halo_mass_msun'] = df_run['halo_mass'] * MASS_CONVERSION_FACTOR
# Shared velocity grid lives in df.attrs, not in the rows (new schema).
if 'v_grid' not in df_run.attrs:
    raise KeyError("df.attrs has no 'v_grid' -- file not written by the current pipeline.")
v_grid = np.asarray(df_run.attrs['v_grid'])

in_bin = df_run[(df_run['halo_mass_msun'] >= 10**MASS_LO_LOG) &
                (df_run['halo_mass_msun'] <  10**MASS_HI_LOG)]
halo_ids_in_bin = np.sort(in_bin['halo_id'].unique())
if len(halo_ids_in_bin) == 0:
    raise ValueError(f"No halo in 10^{MASS_LO_LOG}-10^{MASS_HI_LOG} M_sun in this run.")
if HALO_ID is None:
    HALO_ID = int(np.random.default_rng(SELECT_SEED).choice(halo_ids_in_bin))

df_halo        = df_run[df_run['halo_id'] == HALO_ID]
halo_mass_msun = df_halo['halo_mass_msun'].iloc[0]
print(f"Mass bin 10^{MASS_LO_LOG}-10^{MASS_HI_LOG}: {len(halo_ids_in_bin)} halos")
print(f"  -> halo_id = {HALO_ID} | log M_200 = {np.log10(halo_mass_msun):.2f} "
      f"| {len(df_halo)} skewers")

# ------------------------------------------------------------------
# STEP 2. Snap each requested b onto the real grid and grab its 12 skewers
# ------------------------------------------------------------------
# The grid is 0.0 plus 30 log-spaced values in [0.01, 50], so round numbers are
# usually NOT on it. We match the RAW r_multiplier (r_mult_round would merge
# 0.0100/0.0134 and 0.0180/0.0241 and return 24 skewers per set).
avail_b = np.sort(df_halo['r_multiplier'].unique())
avail_b = avail_b[avail_b > 0]
selected = []
for target_b, color in B_SETS:
    b_exact = float(avail_b[np.argmin(np.abs(avail_b - target_b))])
    if any(np.isclose(b_exact, s['b_exact']) for s in selected):
        raise ValueError(f"b={target_b} snapped onto {b_exact:.4f}, already in use. "
                         f"Available: {np.round(avail_b, 4)}")
    d = df_halo[np.isclose(df_halo['r_multiplier'], b_exact)]
    selected.append({'target': target_b, 'b_exact': b_exact, 'color': color, 'df': d})
    print(f"  b = {target_b} -> grid {b_exact:.4f} R_200 "
          f"({d['impact_kpc'].iloc[0]:.2f} ckpc/h) | {len(d)} skewers "
          f"| HCD: {int(d['is_hcd'].sum())}/{len(d)}")

# ------------------------------------------------------------------
# STEP 3. Halo position from the FoF catalogue
# ------------------------------------------------------------------
# Same logic as utils.HaloFinder: halo_id is the GLOBAL index across the group
# chunks, i.e. exactly the object the pipeline sampled.
group_dir = os.path.join(STORAGE, f'groups_{SNAP_NUM:03d}')
with h5py.File(os.path.join(group_dir, f'fof_subhalo_tab_{SNAP_NUM:03d}.0.hdf5'), 'r') as f:
    hdr = f['Header'].attrs
    n_group_files = int(hdr.get('NumFiles', hdr.get('NumFilesPerSnapshot', 1)))
    BOX, A_SC, HUB = float(hdr['BoxSize']), float(hdr['Time']), float(hdr['HubbleParam'])

offset, halo_props = 0, None
for i in range(n_group_files):
    gf = os.path.join(group_dir, f'fof_subhalo_tab_{SNAP_NUM:03d}.{i}.hdf5')
    if not os.path.exists(gf):
        continue
    with h5py.File(gf, 'r') as f:
        if 'Group' not in f or 'Group_M_Crit200' not in f['Group']:
            continue
        n_here = len(f['Group']['Group_M_Crit200'])
        if HALO_ID < offset + n_here:
            j = int(HALO_ID - offset)
            halo_props = dict(cm=np.array(f['Group']['GroupPos'][j], dtype=float),
                              r200=float(f['Group']['Group_R_Crit200'][j]),
                              mass=float(f['Group']['Group_M_Crit200'][j]))
            break
        offset += n_here
if halo_props is None:
    raise ValueError(f"halo_id {HALO_ID} not found in the group catalogue.")

CM, R200 = halo_props['cm'], halo_props['r200']
HALF = BOX_HALF_R200 * R200                       # map half-width, ckpc/h
print(f"\nHalo centre = {np.round(CM,1)} ckpc/h | R_200 = {R200:.2f} ckpc/h")
print(f"Map = +-{HALF:.1f} ckpc/h ({2*HALF/HUB*A_SC:.1f} physical kpc across)")

# ------------------------------------------------------------------
# STEP 4. Read the gas cells inside the box (cached across re-runs)
# ------------------------------------------------------------------
# Reading 128 chunks takes minutes; the cache lets you re-run this cell to
# tweak the figure without touching the disk again.
_cache_key = (SIM_NAME, SNAP_NUM, HALO_ID, BOX_HALF_R200, MAX_CHUNKS)
if USE_CACHE and '_gas_cache' in globals() and _gas_cache.get('key') == _cache_key:
    pos_box, mass_box, fhi_box, has_hi = (_gas_cache['pos'], _gas_cache['mass'],
                                          _gas_cache['fhi'], _gas_cache['has_hi'])
    print(f"\nUsing cached gas cells ({len(pos_box):,}).")
else:
    snap_dir = os.path.join(STORAGE, f'snapdir_{SNAP_NUM:03d}')
    with h5py.File(os.path.join(snap_dir, f'snap_{SNAP_NUM:03d}.0.hdf5'), 'r') as f:
        n_chunks = int(f['Header'].attrs['NumFilesPerSnapshot'])
    n_scan = n_chunks if MAX_CHUNKS is None else min(MAX_CHUNKS, n_chunks)
    print(f"\nScanning {n_scan}/{n_chunks} snapshot chunks...")

    pos_box, mass_box, fhi_box, has_hi = [], [], [], True
    for i in range(n_scan):
        sf = os.path.join(snap_dir, f'snap_{SNAP_NUM:03d}.{i}.hdf5')
        if not os.path.exists(sf):
            continue
        with h5py.File(sf, 'r') as f:
            if 'PartType0' not in f:
                continue
            g = f['PartType0']
            d = g['Coordinates'][:] - CM
            d -= BOX * np.round(d / BOX)          # minimum image: safe at box edges
            m = np.all(np.abs(d) <= HALF, axis=1)
            if not m.any():
                continue
            pos_box.append(d[m])
            mass_box.append(g['Masses'][:][m])    # 1e10 M_sun/h
            if 'NeutralHydrogenAbundance' in g:
                fhi_box.append(g['NeutralHydrogenAbundance'][:][m])
            else:
                has_hi = False
        if (i + 1) % 16 == 0 or i == n_scan - 1:
            print(f"  chunk {i+1:>3}/{n_scan} | {sum(len(p) for p in pos_box):,} cells so far")

    if not pos_box:
        raise RuntimeError("No gas cell in the box -- check STORAGE / SNAP_NUM.")
    pos_box  = np.concatenate(pos_box)
    mass_box = np.concatenate(mass_box)
    fhi_box  = np.concatenate(fhi_box) if has_hi else np.ones(len(mass_box))
    print(f"  -> {len(pos_box):,} gas cells inside the box")
    if not has_hi:
        print("  ! NeutralHydrogenAbundance absent: showing TOTAL hydrogen instead")
    _gas_cache = dict(key=_cache_key, pos=pos_box, mass=mass_box,
                      fhi=fhi_box, has_hi=has_hi)

# ------------------------------------------------------------------
# STEP 5. Project along z -> HI column density [cm^-2]
# ------------------------------------------------------------------
# Nearest-grid-point deposition of each cell's HI mass: a mass histogram, not a
# Voronoi rendering, so pixel values are indicative and will NOT match the
# sightline N_HI cell for cell (the map has all HI, the skewers only the HCD).
edges     = np.linspace(-HALF, HALF, NPIX + 1)
m_hi_msun = mass_box * 1e10 / HUB * X_HYDROGEN * fhi_box
H, _, _   = np.histogram2d(pos_box[:, 0], pos_box[:, 1],
                           bins=[edges, edges], weights=m_hi_msun)
if SMOOTH_SIGMA > 0:
    H = gaussian_filter(H, SMOOTH_SIGMA)
# pixel side in PHYSICAL cm: ckpc/h -> /h -> comoving kpc -> x a -> cm
pix_cm = (2 * HALF / NPIX) / HUB * A_SC * KPC_CM
N_HI   = H * MSUN_G / MPROTON_G / pix_cm**2
logN   = np.log10(np.where(N_HI > 0, N_HI, np.nan))
print(f"Map log N_HI: {np.nanmin(logN):.2f} .. {np.nanmax(logN):.2f}")


In [ ]:
# ------------------------------------------------------------------
# STEP 6. Where each skewer falls on a z projection
# ------------------------------------------------------------------
# Reconstructs radii_skewers.generate_skewers exactly. Projecting along z:
#   axis == 2 (LOS || z) -> perpendicular to the map  -> a POINT
#   axis == 0 (LOS || x) -> runs inside the map plane -> a HORIZONTAL line
#   axis == 1 (LOS || y) -> ditto                     -> a VERTICAL line
# The Top/Bottom pair of axes 0 and 1 is offset along z, i.e. along the very
# direction we project, so it carries no information in this view and would
# just pile up on the central line: we skip it. That leaves 8 of the 12
# sightlines drawn per radius (4 points + 2 horizontal + 2 vertical); the
# bottom panel still shows all 12.
def _offset_vec(axis, tag, impact):
    o1, o2 = {'Right': (impact, 0.0), 'Left': (-impact, 0.0),
              'Top': (0.0, impact), 'Bottom': (0.0, -impact)}[tag]
    vec, free = np.zeros(3), [k for k in range(3) if k != axis]
    vec[free[0]], vec[free[1]] = o1, o2
    return vec

# ------------------------------------------------------------------
# STEP 7. The figure: map on top, spectra below
# ------------------------------------------------------------------
# Layout notes: the figure is deliberately large so nothing has to be shrunk.
# Both legends sit ABOVE their panel, outside the axes, so they can never cover
# the data; tick locators are set explicitly so the numbers stop colliding.
fig = plt.figure(figsize=(13, 19))
gs  = fig.add_gridspec(2, 1, height_ratios=[1.3, 1.0], hspace=0.16,
                       top=0.94, bottom=0.045)
ax_map, ax_flux = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

# --- 7a. the map ---
im = ax_map.imshow(logN.T, origin='lower', cmap='inferno',
                   extent=[-HALF/R200, HALF/R200, -HALF/R200, HALF/R200],
                   vmin=15.0, vmax=np.nanmax(logN), interpolation='nearest')
ax_map.set_aspect('equal')
cax = make_axes_locatable(ax_map).append_axes("right", size="4%", pad=0.10)
cb  = fig.colorbar(im, cax=cax)
cb.set_label(r'$\log N_{\rm HI}\ [\mathrm{cm^{-2}}]$', fontsize=17)

# HCD (17.2) and DLA (20.3) isocontours: the same thresholds used everywhere
# else in this notebook. clabel's fmt dict replaces the number by the regime.
centres = 0.5 * (edges[:-1] + edges[1:]) / R200
cs = ax_map.contour(centres, centres, np.nan_to_num(logN, nan=0.0).T,
                    levels=[17.2, 20.3], colors=['deepskyblue', 'white'],
                    linewidths=[1.2, 1.6], linestyles=['--', '-'])
ax_map.clabel(cs, fmt={17.2: 'HCD', 20.3: 'DLA'}, fontsize=10)

ax_map.add_patch(plt.Circle((0, 0), 1.0, fill=False, color='lime',
                            lw=1.8, ls='--', zorder=5))
ax_map.text(0.72, 0.72, r'$R_{200}$', color='lime', fontsize=13, zorder=6)

# --- 7b. the sightlines on the map (in-plane ones only, see STEP 6) ---
for s in selected:
    b_code = s['b_exact'] * R200
    ax_map.add_patch(plt.Circle((0, 0), s['b_exact'], fill=False, color=s['color'],
                                lw=1.0, ls=':', alpha=0.85, zorder=5))
    for _, row in s['df'].iterrows():
        ax_i = int(row['axis'])
        vec  = _offset_vec(ax_i, row['position_tag'], b_code) / R200
        if ax_i == 2:                                    # LOS along z -> point
            ax_map.plot(vec[0], vec[1], marker='o', ms=10, mfc=s['color'],
                        mec='white', mew=1.5, zorder=9)
        elif np.isclose(vec[2], 0.0):                    # in the map plane -> line
            if ax_i == 0:
                ax_map.axhline(vec[1], color=s['color'], lw=1.3, alpha=0.75, zorder=4)
            else:
                ax_map.axvline(vec[0], color=s['color'], lw=1.3, alpha=0.75, zorder=4)
        # else: offset along z -> not shown (degenerate in this projection)

ax_map.set_xlim(-HALF/R200, HALF/R200); ax_map.set_ylim(-HALF/R200, HALF/R200)
ax_map.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax_map.yaxis.set_major_locator(ticker.MultipleLocator(1))
ax_map.set_xlabel(r'$\Delta x\ [R_{200}]$')
ax_map.set_ylabel(r'$\Delta y\ [R_{200}]$')
ax_map.set_title(rf'Halo {HALO_ID} — HI column density, projected along $z$'
                 rf'  ($\log M_{{200}}={np.log10(halo_mass_msun):.2f}$, {SIM_NAME})',
                 fontsize=17, pad=50)          # pad leaves room for the legend below it

h_map  = [Line2D([0], [0], marker='o', ls='none', mfc=s['color'], mec='white', ms=10,
                 label=rf"$b={s['b_exact']:.3f}\,R_{{200}}$  (LOS $\parallel z$)")
          for s in selected]
h_map += [Line2D([0], [0], color='gray', lw=1.6,
                 label=r'LOS $\parallel x,y$  (in the plane)')]
ax_map.legend(handles=h_map, loc='lower center', bbox_to_anchor=(0.5, 1.015),
              ncol=3, fontsize=12, framealpha=0.95, columnspacing=1.5,
              edgecolor='lightgray')

# --- 7c. the spectra of those same skewers (all 12 per radius) ---
# HCD-mode run: sightlines below the 10^17.2 cut were zeroed and are flat at F=1.
for s in selected:
    for _, row in s['df'].iterrows():
        ax_flux.plot(v_grid, np.asarray(row['flux'], dtype=float),
                     color=s['color'], lw=1.0, alpha=0.65, zorder=3)
    if SHOW_MEDIAN:
        stack = np.vstack([np.asarray(f, dtype=float) for f in s['df']['flux']])
        ax_flux.plot(v_grid, np.median(stack, axis=0), color=s['color'], lw=2.6, zorder=6)

ax_flux.axvline(0.0, color='gray', ls=':', lw=1.5, alpha=0.8, zorder=1)
ax_flux.axhline(1.0, color='gray', ls='--', lw=1.0, alpha=0.6, zorder=1)
if VEL_WINDOW:
    ax_flux.set_xlim(-VEL_WINDOW, VEL_WINDOW)
ax_flux.set_ylim(-0.05, 1.15)
# Explicit locators: the default put a label every 500 km/s and they collided.
ax_flux.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10]))
ax_flux.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
ax_flux.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax_flux.set_xlabel(r'Velocity relative to halo $[\mathrm{km\ s^{-1}}]$')
ax_flux.set_ylabel(r'Normalized Flux $F = e^{-\tau}$')
ax_flux.grid(True, ls='--', alpha=0.4)

h_flux = [Line2D([0], [0], color=s['color'], lw=2.0,
                 label=rf"$b={s['b_exact']:.3f}\,R_{{200}}$ ({len(s['df'])} skewers, "
                       rf"{int(s['df']['is_hcd'].sum())} HCD)") for s in selected]
if SHOW_MEDIAN:
    h_flux.append(Line2D([0], [0], color='gray', lw=2.6, label='Median of each set'))
ax_flux.legend(handles=h_flux, loc='lower center', bbox_to_anchor=(0.5, 1.015),
               ncol=len(h_flux), fontsize=12, framealpha=0.95, edgecolor='lightgray')

# --- 7d. align the two panels ---
# The colorbar plus the equal aspect ratio shrink the map, so match the bottom
# panel to the map's real width instead of letting gridspec keep them different.
fig.canvas.draw()
p_map, p_flux = ax_map.get_position(), ax_flux.get_position()
ax_flux.set_position([p_map.x0, p_flux.y0, p_map.width, p_flux.height])
plt.show()